# 运行时：我有哪些选择？该如何选择？

请记住，TensorRT 由两个主要组件构成——**1. 一系列解析器和集成工具**，用于将模型转换为优化后的引擎；以及 **2. 一系列 TensorRT 运行时 API**，并附带多种用于部署的相关工具。

在本 Notebook 中，我们将重点关注后者——TensorRT 引擎的各种运行时选项。

不同的运行时适用于运行 TRT 引擎的不同用例。

### 选择运行时时的考量因素：

一般来说，在选择运行时有几个主要的考量因素：
- **框架** - 某些选项（如 Torch-TRT）仅与 PyTorch 相关。
- **解决问题的时间** - 如果需要快速解决方案且 ONNX 转换失败，Torch-TRT 更有可能“开箱即用”。
- **服务需求** - Torch-TRT 可以使用 TorchServe 通过云简单地提供模型服务。对于其他框架（或需要更高级的功能），TRITON 是框架无关的，允许并发模型执行或在一个 GPU 内运行多个副本以降低延迟，并且可以接受通过 ONNX 和 Torch-TRT 路径创建的引擎。
- **性能** - 不同的 TensorRT 运行时提供不同水平的性能。例如，Torch-TRT 通常比直接使用 ONNX 或 C++ API 慢。

### Python API：

**适用于以下情况：**
- 你可以接受一定的性能开销，且
- 你对 Python 最为熟悉，或
- 你正在进行 TRT 的初步调试和测试

**更多信息：**

https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#perform_inference_python 让你能够通过 Python 接口精细控制引擎的执行。它显式地处理内存分配、内核执行以及 GPU 与主机之间的数据拷贝——这使得将其集成到高性能应用程序中更加容易。它也非常适合在 Python 环境（例如 Jupyter Notebook）中测试模型。

./2. Using PyTorch through ONNX.ipynb 是一个很好的示例，展示了如何在使用 Python 的同时利用 TensorRT 获得出色的性能。

### C++ API：

**适用于以下情况：**
- 你想要尽可能少的开销，以最大化模型性能并获得更好的延迟
- 你没有使用 Torch-TRT（尽管仅生成单个引擎的 Torch-TRT 图转换仍然可以导出到 C++）
- 你对 C++ 最为熟悉
- 你想要尽可能优化你的推理流水线

**更多信息：**

https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#perform_inference_c 让你能够通过 C++ 接口精细控制引擎的执行。它显式地处理内存分配、内核执行以及 GPU 与主机之间的数据拷贝——这使得将其集成到高性能 C++ 应用程序中更加容易。C++ API 通常是运行 TensorRT 引擎性能最高的选项，开销最小。

https://developer.nvidia.com/blog/speed-up-inference-tensorrt/ 是一个很好的示例，展示了如何使用 C++ API 运行带有动态批量大小支持的 ONNX 模型。

### Torch-TRT 运行时：

**适用于以下情况：**
- 你正在使用 Torch-TRT，且
- 你的模型被转换为多个 TensorRT 引擎

**更多信息：**

Torch-TRT 是与通过 Torch-TRT 转换的模型一起使用的标准运行时。它的工作原理是一次获取 PyTorch 图中的一组节点，并用一个单一的优化引擎替换它们，该引擎在后台调用 TensorRT Python API。这个优化后的引擎以 PyTorch 操作的形式存在——这意味着你的计算图仍然在 PyTorch 中，并且本质上像任何其他 PyTorch 模型一样运行。

如果你的图完全转换为单个 Torch-TRT 引擎，导出该引擎节点并使用其他 API 之一运行可能会更高效。你可以在 https://pytorch.org/TensorRT/index.html 中找到执行此操作的说明。

作为示例，本指南随附的 Torch-TRT Notebook 使用了 Torch-TRT 运行时。

### TRITON Inference Server

**适用于以下情况：**
- 你想通过 HTTP 或 gRPC 提供模型服务
- 你想在多个模型或多个模型副本之间进行负载均衡，以最小化延迟并更好地利用 GPU
- 你想让多个模型同时在单个 GPU 上高效运行
- 你想通过一个统一的接口，提供由各种转换器和框架（包括 Torch-TRT 和 ONNX）转换的各种模型
- 你需要服务支持，但使用的是 PyTorch、其他框架，或是通用的 ONNX 路径

**更多信息：**

TRITON 是一款开源推理服务软件，允许团队在任何基于 GPU 或 CPU 的基础设施（云、数据中心或边缘）上部署来自任何框架（TensorFlow、TensorRT、PyTorch、ONNX Runtime 或自定义框架）的训练好的 AI 模型，模型可以存储在本地存储、Google Cloud Platform 或 AWS S3 上。它是一个灵活的项目，具有多项独特功能——例如异构模型和同一模型的多个副本的并发模型执行（多个模型副本可以进一步减少延迟），以及负载均衡和模型分析。如果你需要通过 HTTP 提供模型服务（例如在云推理解决方案中），它是一个不错的选择。

You can find the TRITON home page [here](https://developer.nvidia.com/triton-inference-server), and the documentation [here](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/).



#  Runtimes: What are my options? How do I choose?

Remember that TensorRT consists of two main components - __1. A series of parsers and integrations__ to convert your model to an optimized engine and __2. An series of TensorRT runtime APIs__ with several associated tools for deployment.

In this notebook, we will focus on the latter - various runtime options for TensorRT engines.

The runtimes have different use cases for running TRT engines. 

### Considerations when picking a runtime:

Generally speaking, there are a few major considerations when picking a runtime:
- __Framework__ - Some options, like Torch-TRT, are only relevant to PyTorch
- __Time-to-solution__ - Torch-TRT is much more likely to work 'out-of-the-box' if a quick solution is required and ONNX fails
- __Serving needs__ - Torch-TRT can use TorchServe to serve models over Cloud as a simple solution. For other frameworks (or for more advanced features) TRITON is framework agnostic, allows for concurrent model execution or multiple copies within a GPU to reduce latency, and can accept engines created through both the ONNX and Torch-TRT paths
- __Performance__ - Different TensorRT runtimes offer varying levels of performance. For example, Torch-TRT is generally going to be slower than using ONNX or the C++ API directly.

### Python API:

__Use this when:__
- You can accept some performance overhead, and
- You are most familiar with Python, or
- You are performing initial debugging and testing with TRT

__More info:__

 
The [TensorRT Python API](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#perform_inference_python) gives you fine-grained control over the execution of your engine using a Python interface. It makes memory allocation, kernel execution, and copies to and from the GPU explicit - which can make integration into high performance applications easier. It is also great for testing models in a Python environment - such as in a Jupyter notebook.
 
The [ONNX notebook for PyTorch](./2.%20Using%20PyTorch%20through%20ONNX.ipynb) is a good example of using TensorRT to get great performance while staying in Python.

### C++ API: 

__Use this when:__
- You want the least amount of overhead possible to maximize the performance of your models and achieve better latency
- You are not using Torch-TRT (though Torch-TRT graph conversions that only generate a single engine can still be exported to C++)
- You are most familiar with C++
- You want to optimize your inference pipeline as much as possible

__More info:__

The [TensorRT C++ API](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#perform_inference_c) gives you fine-grained control over the execution of your engine using a C++ interface. It makes memory allocation, kernel execution, and copies to and from the GPU explicit - which can make integration into high performance C++ applications easier. The C++ API is generally the most performant option for running TensorRT engines, with the least overhead.

[This NVIDIA Developer blog](https://developer.nvidia.com/blog/speed-up-inference-tensorrt/) is a good example of taking an ONNX model and running it with dynamic batch size support using the C++ API.


### Torch-TRT Runtime:
    
__Use this when:__
    
- You are using Torch-TRT, and
- Your model converts to more than one TensorRT engine

__More info:__


Torch-TRT is the standard runtime used with models that were converted in Torch-TRT. It works by taking groups of nodes at once in the PyTorch graph, and replacing them with a singular optimized engine that calls the TensorRT Python API behind the scenes. This optimized engine is in the form of a PyTorch operation - which means that your graph is still in PyTorch and will essentially function like any other PyTorch model.

If your graph entirely converts to a single Torch-TRT engine, it can be more efficient to export the engine node and run it using one of the other APIs. You can find instructions to do this in the [Torch-TRT documentation](https://pytorch.org/TensorRT/index.html).

As an example, the Torch-TRT notebooks included with this guide use the Torch-TRT runtime.

###  TRITON Inference Server

__Use this when:__
- You want to serve your models over HTTP or gRPC
- You want to load balance across multiple models or copies of models across GPUs to minimze latency and make better use of the GPU
- You want to have multiple models running efficiently on a single GPU at the same time
- You want to serve a variety of models converted using a variety of converters and frameworks (including Torch-TRT and ONNX) through a uniform interface
- You need serving support but are using PyTorch, another framework, or the ONNX path in general

__More info:__


TRITON is an open source inference serving software that lets teams deploy trained AI models from any framework (TensorFlow, TensorRT, PyTorch, ONNX Runtime, or a custom framework), from local storage or Google Cloud Platform or AWS S3 on any GPU- or CPU-based infrastructure (cloud, data center, or edge). It is a flexible project with several unique features - such as concurrent model execution of both heterogeneous models and multiple copies of the same model (multiple model copies can reduce latency further) as well as load balancing and model analysis. It is a good option if you need to serve your models over HTTP - such as in a cloud inferencing solution.
    
You can find the TRITON home page [here](https://developer.nvidia.com/triton-inference-server), and the documentation [here](https://docs.nvidia.com/deeplearning/triton-inference-server/user-guide/docs/).